# RAG walkthrough — PubMedQA + bge-small + Chroma + Claude

A runnable, cell-by-cell tour of the pipeline in this repo. Run top to bottom.

**Two halves, kept separate on purpose:**

```
question -> embed(bge-small) -> top-k from Chroma -> Claude(+citations) -> answer
           \____________ RETRIEVAL (local, $0) ___________/  \__ GENERATION __/
```

- **Retrieval** needs no API key (embeddings run locally).
- **Generation & eval** need `ANTHROPIC_API_KEY` in the environment.

> Run this notebook with the project venv's kernel: `.venv/bin/python -m ipykernel ...`,
> or just `.venv/bin/jupyter lab`. The cells below use `sys.executable` so subprocess
> calls reuse whatever Python is running the kernel.

## 0. Setup — imports & environment check

In [ ]:
import os, sys, subprocess
import config

print('Python   :', sys.executable)
print('Corpus   :', config.DATASET_NAME, config.DATASET_CONFIG)
print('Embedder :', config.EMBED_MODEL)
print('LLM      :', config.CLAUDE_MODEL)
print('Chroma   :', config.CHROMA_DIR)
print('API key  :', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'NOT set (generation cells will skip)')

## 1. Config — the single source of truth

Every script imports from `config.py`, so ingest / query / eval can never disagree
about model, chunk size, or collection. Two RAG gotchas live here:

- **`QUERY_PREFIX`** — bge was trained with an instruction prefix on *queries only*.
- **`CHUNK_MAX_TOKENS = 500`** — bge silently truncates past 512 tokens.

In [ ]:
for k in ['DATASET_CONFIG','EMBED_MODEL','QUERY_PREFIX','CHUNK_MAX_TOKENS',
          'CHUNK_OVERLAP_SENTENCES','COLLECTION_NAME','TOP_K','CLAUDE_MODEL','MAX_TOKENS']:
    print(f'{k:>24} = {getattr(config, k)!r}')

## 2. The embedder — text becomes vectors

`normalize_embeddings=True` returns unit vectors, so dot product == cosine similarity —
which is the space we tell Chroma to use. Note the query gets the prefix; the document does not.

In [ ]:
from embedder import embed_query, embed_documents
import numpy as np

q_vec = embed_query('Is iron deficiency associated with restless legs syndrome?')
d_vec = embed_documents(['Iron deficiency has been linked to restless legs syndrome in several studies.'])[0]

print('embedding dim        :', len(q_vec))            # 384 for bge-small
print('query vec is unit-len:', round(float(np.linalg.norm(q_vec)), 4))
print('cosine(query, doc)   :', round(float(np.dot(q_vec, d_vec)), 4))  # higher = more similar

## 3. Chunking — sentence-aware, measured with the real tokenizer

Greedily pack whole sentences until the next would exceed `CHUNK_MAX_TOKENS`, then flush —
carrying `CHUNK_OVERLAP_SENTENCES` across the boundary so a split fact stays retrievable.
Most PubMed abstracts fit in one chunk; the demo text below is padded to force a split.

In [ ]:
from ingest import chunk_text, split_sentences

long_text = ' '.join(f'This is sentence number {i} describing a distinct clinical finding.' for i in range(80))
chunks = chunk_text(long_text)
print(f'{len(split_sentences(long_text))} sentences -> {len(chunks)} chunks')
for i, c in enumerate(chunks):
    print(f'  chunk {i}: {len(c)} chars, starts: {c[:60]!r}')

## 4. Ingest — download -> chunk -> embed -> store (run once)

Local embeddings only, so **$0 in API cost**. Idempotent: re-running with the same
or smaller `--limit` does nothing. Start small with 50; drop `--limit` for all 1,000.

In [ ]:
proc = subprocess.run([sys.executable, 'ingest.py', '--limit', '50'],
                      capture_output=True, text=True)
print(proc.stdout[-800:])
if proc.returncode != 0:
    print('STDERR:', proc.stderr[-800:])

## 5. Retrieval — embed the question, pull top-k (NO API key needed)

This is the pure RAG core. Distance is `1 - cosine similarity`, so **lower = closer**.

In [ ]:
from query import get_collection, retrieve

collection = get_collection()
question = 'Is adjustment for reporting heterogeneity necessary in sleep disorders?'
hits = retrieve(collection, question)
print('Q:', question, '\n')
for i, (text, meta, dist) in enumerate(hits):
    print(f'  [{i}] pubid={meta["pubid"]:>10}  cos_dist={dist:.3f}  {text[:80].strip()}...')

## 6. Honest retrieval check — does a question find its OWN abstract?

This is exactly what `eval.py`'s **hit-rate** aggregates: pick a question whose abstract
we ingested, and check whether a chunk with its `pubid` comes back in the top-k.
Watch the gold doc land at rank 0 with a distance clearly below the rest.

In [ ]:
from datasets import load_dataset

ds = load_dataset(config.DATASET_NAME, config.DATASET_CONFIG, split=config.DATASET_SPLIT).select(range(50))
row = ds[7]
q, gold = row['question'], str(row['pubid'])
hits = retrieve(collection, q)
print('Q   :', q)
print('GOLD:', gold, '\n')
hit = False
for i, (text, meta, dist) in enumerate(hits):
    is_gold = meta['pubid'] == gold
    hit = hit or is_gold
    print(f'  [{i}] pubid={meta["pubid"]:>10}  cos_dist={dist:.3f}' + ('   <-- GOLD' if is_gold else ''))
print('\nHIT@5:', hit)

## 7. Generation with citations (needs `ANTHROPIC_API_KEY`)

Each retrieved chunk goes to Claude as a `document` block with `citations.enabled=True`,
so the answer comes back split into text blocks whose `.citations` point at the exact
source chunk and character span. Skips automatically if no key is set.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('No ANTHROPIC_API_KEY set - skipping generation. Set it and re-run this cell.')
else:
    from query import answer, print_response
    q = 'Is adjustment for reporting heterogeneity necessary in sleep disorders?'
    hits = retrieve(collection, q)
    resp = answer(q, hits)
    print_response(resp, hits)

## 8. Eval — two metrics that mean different things

- **Retrieval hit-rate**: did the question's own gold abstract come back? (retriever only, $0)
- **Decision accuracy**: does Claude's yes/no/maybe match the gold label? (end-to-end proxy)

Diagnostic: *low hit-rate -> fix retrieval; high hit-rate but low accuracy -> fix the prompt/model.*
Keep `--n <= the --limit you ingested with`. Needs the API key.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('No ANTHROPIC_API_KEY set - skipping eval. Set it and re-run this cell.')
else:
    proc = subprocess.run([sys.executable, 'eval.py', '--n', '5'],
                          capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-800:])

## Next steps (the Adelphi pivot)

You now understand the baseline hands-on. From here the roadmap is:

1. **Pluggable model layer** — one interface, Claude now / local LLM for air-gap.
2. **Agentic + MCP** — expose retrieval as tools an agent calls via an MCP server.
3. **Federated + access control** — multiple siloed corpora, permission-aware retrieval
   (mirrors Adelphi's Connector product).
4. **Explainability & security** — grounding checks, citation verification, refusal, audit.